# ASR Performance Evaluation: Whisper-tiny on FLEURS (Azerbaijani)

This project evaluates the accuracy of Automatic Speech Recognition (ASR) for the Azerbaijani language using the OpenAI Whisper-tiny model against the Google FLEURS dataset.





To run this pipeline, you must install specific libraries for audio processing (librosa), model handling (transformers, datasets), and performance evaluation (evaluate, jiwer).

In [ ]:
!pip install -q "datasets<3.0.0" transformers evaluate jiwer librosa pandas torch

## Whisper Fine-Tuning Pipeline (Azerbaijani)



This module handles the supervised fine-tuning of the Whisper-tiny model on the Azerbaijani (az_az) subset of the Google FLEURS dataset. It includes data preprocessing, dynamic batching, and trainer configuration

#
#
#
#

#### *Data Acquisition and Preprocessing*

> The pipeline begins by loading the training and validation splits. To ensure compatibility with the Whisper architecture, the audio is resampled to 16,000 Hz.

#### *Tokenization and Feature Extraction*

> The prepare_dataset function transforms raw audio and text into a format suitable for the mode

#### *Data Collator*

> The DataCollatorSpeechSeq2SeqWithPadding class is critical for handling variable-length sequences within a single batch


#### *Evaluation Metrics*

> The compute_metrics function evaluates model performance during training using the Word Error Rate (WER)


#### *Training Configuration and Execution*

> This section defines the hyperparameters and executes the training loop using the Seq2SeqTraine


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("1. Fine-Tuning")
train_dataset = load_dataset("google/fleurs", "az_az", split="train[:150]", trust_remote_code=True)
eval_dataset = load_dataset("google/fleurs", "az_az", split="validation[:50]", trust_remote_code=True)

train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
eval_dataset = eval_dataset.cast_column("audio", Audio(sampling_rate=16000))

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

print("2.Tokenization and feature extraction")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc=1)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=eval_dataset.column_names, num_proc=1)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]

        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)



def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

print("3.Setting up training parameters")
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-az-finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=10,
    max_steps=50,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=25,
    eval_steps=25,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("4.Start of Fine Tuning!")
trainer.train()

#### *Visualization of Training Metrics*


> This final block transforms the numerical logs from the training process into a visual format. It provides a clear overview of how the model's error rates and loss values evolved over the training steps

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = [log['step'] for log in logs if 'loss' in log]
train_loss = [log['loss'] for log in logs if 'loss' in log]

eval_steps = [log['step'] for log in logs if 'eval_loss' in log]
eval_loss = [log['eval_loss'] for log in logs if 'eval_loss' in log]
eval_wer = [log['eval_wer'] * 100 for log in logs if 'eval_wer' in log] # Сразу в проценты

fig, ax1 = plt.subplots(figsize=(8, 5))

color = 'tab:blue'
ax1.set_xlabel('Training Steps')
ax1.set_ylabel('Loss', color=color)
ax1.plot(train_steps, train_loss, label='Training Loss', marker='o', color='blue', linewidth=2)
ax1.plot(eval_steps, eval_loss, label='Validation Loss', marker='s', color='orange', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(eval_steps)
ax1.grid(True, linestyle='--', alpha=0.7)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('WER (%)', color=color)
ax2.plot(eval_steps, eval_wer, label='WER (%)', marker='^', color='green', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

plt.title('Fine-Tuning Progress: Loss and WER')
fig.tight_layout()
plt.savefig('training_graph.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
print('Loading test set (same as Part A)')
test_dataset_b3 = load_dataset('google/fleurs', 'az_az', split='test[:200]', trust_remote_code=True)
test_dataset_b3 = test_dataset_b3.cast_column('audio', Audio(sampling_rate=16000))
print(f'Test samples: {len(test_dataset_b3)}')


#### *Inference Helper and Evaluation Metrics*
> `run_inference_b3` runs transcription for any given model and processor on the full test set, enabling a fair comparison under identical conditions. WER and CER metrics are loaded once and reused across both evaluations.

In [ ]:
import evaluate
wer_metric_b3 = evaluate.load('wer')
cer_metric_b3 = evaluate.load('cer')

def run_inference_b3(mdl, proc, dataset):
    mdl.eval()
    forced_ids = proc.get_decoder_prompt_ids(language='az', task='transcribe')
    preds, refs = [], []
    with torch.no_grad():
        for item in dataset:
            audio = item['audio']
            features = proc(
                audio['array'],
                sampling_rate=audio['sampling_rate'],
                return_tensors='pt'
            ).input_features.to(device)
            ids = mdl.generate(features, forced_decoder_ids=forced_ids)
            preds.append(proc.batch_decode(ids, skip_special_tokens=True)[0].strip())
            refs.append(item['transcription'].strip())
    return preds, refs

#### *Base Model Evaluation*
> The original `openai/whisper-tiny` checkpoint is loaded fresh and evaluated on the same 200-sample test set used in Part A to establish the pre-fine-tuning baseline.

In [ ]:
print('Loading base model')
base_model_b3 = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny').to(device)

print('Running inference (base)')
base_preds, refs_b3 = run_inference_b3(base_model_b3, processor, test_dataset_b3)

base_wer = wer_metric_b3.compute(predictions=base_preds, references=refs_b3) * 100
base_cer = cer_metric_b3.compute(predictions=base_preds, references=refs_b3) * 100
print(f'Base  →  WER: {base_wer:.2f}%  |  CER: {base_cer:.2f}%')

#### *Fine-Tuned Model Evaluation*
> The fine-tuned model — already in memory after `trainer.train()` — is evaluated on the same test set to measure the effect of fine-tuning under directly comparable conditions.

In [ ]:
print('Running inference (fine-tuned)...')
ft_preds, _ = run_inference_b3(model, processor, test_dataset_b3)

ft_wer = wer_metric_b3.compute(predictions=ft_preds, references=refs_b3) * 100
ft_cer = cer_metric_b3.compute(predictions=ft_preds, references=refs_b3) * 100
print(f'Fine-tuned  →  WER: {ft_wer:.2f}%  |  CER: {ft_cer:.2f}%')

#### *Comparison Table*
> WER and CER scores for both models are summarised in a single table with delta values (Δ) showing the change relative to the base model. A negative Δ indicates improvement; a positive Δ indicates regression.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Model':         ['Whisper-tiny (base)', 'Whisper-tiny (fine-tuned)'],
    'WER (%)':       [round(base_wer, 2), round(ft_wer, 2)],
    'CER (%)':       [round(base_cer, 2), round(ft_cer, 2)],
    'WER Δ':         ['-', f'{ft_wer - base_wer:+.2f}%'],
    'CER Δ':         ['-', f'{ft_cer - base_cer:+.2f}%'],
})

print('\n' + '='*55)
print('         Model Comparison — Part B3')
print('='*55)
print(comparison.to_string(index=False))
print('='*55)
print('Negative Δ = improvement, positive Δ = regression')

comparison.to_csv('b3_comparison.csv', index=False)
print('Saved → b3_comparison.csv')

#### *Visualization*
> A grouped bar chart provides a visual side-by-side comparison of WER and CER for the base and fine-tuned models. The figure is saved to `results/b3_comparison.png`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
x, width = range(2), 0.35
labels = ['Base', 'Fine-tuned']

bars_w = ax.bar([i - width/2 for i in x], [base_wer, ft_wer], width, label='WER (%)', color='steelblue')
bars_c = ax.bar([i + width/2 for i in x], [base_cer, ft_cer], width, label='CER (%)', color='coral')

for bar in [*bars_w, *bars_c]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Error Rate (%)', fontsize=12)
ax.set_title('Base vs Fine-Tuned Whisper-tiny\n(FLEURS az_az, 200 test samples)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('b3_comparison.png', dpi=300)
print('Saved → b3_comparison.png')
plt.show()